In [ ]:
# Third-Party Tools: Expanding Horizons with Tavily and Google ADK
# In this notebook, you'll learn how to expand your ADK agent's capabilities by integrating a third-party tool—Tavily—for live web search.  
# You'll see how to wrap LangChain tools for ADK, configure your agent, and run real queries for up-to-date information in WanderWise.

In [1]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain_community tavily-python --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# (optional) Verify installation
%pip show google-adk
%pip show langchain_community 
%pip show tavily-python

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: langchain-community
Version: 0.3.24
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, reque

In [4]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [5]:
# Import Required Modules
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.langchain_tool import LangchainTool
from google.genai import types
from IPython.display import Markdown, display

In [6]:
# Tavily Search API: Set Up
# API Key Management: Always use environment variables or a secure vault for credentials!
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY environment variable not set. "
        "Never hardcode API keys in your code. "
        "Set it securely in your environment or use a secrets manager."
)

In [7]:
# Tavily LangChain tool: Initilaization
# This tool allows the agent to perform live web searches using Tavily.
from langchain_community.tools import TavilySearchResults
tavily_tool_instance = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=True,
    include_images=True,
)

# Wrap it for ADK compatibility
adk_tavily_tool = LangchainTool(tool=tavily_tool_instance)

In [8]:
# Latest Events Agent: Definition
# Define the agent with Tavily as a third-party tool
# The instruction set is the heart of the agent. Make it specific, structured, and safe.
latest_events_agent = Agent(
    name="latest_events_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that uses the Tavily search tool to find and summarize the latest events, festivals, and activities for a given destination, providing up-to-date and user-friendly travelinformation.",
    instruction="""
        You are a travel assistant.
        When a user asks about current or upcoming events, festivals, or activities at a destination, use the Tavily search tool to find the most recent and relevant information.
        Summarize your findings in a clear, concise, and user-friendly way.
        Only use the Tavily tool for event or activity-related queries, and ensure your answers are accurate and helpful.
        If you cannot find relevant information, politely let the user know.
        Always provide the source of your information.
        If the user asks about general travel advice or information not related to current events, politely redirect them to other resources.
        Never expose API keys or sensitive data in your responses.
    """,
    tools=[adk_tavily_tool],
)

In [9]:
# Input Validation Function
import re

def validate_city_query(query: str) -> bool:
    """
    Basic input validation for city/event queries.
    Only allows letters, spaces, and common punctuation.
    Returns True if valid, False otherwise.
    """
    pattern = r"^[\w\s,.'-?!]+$"
    return bool(re.match(pattern, query.strip()))

In [10]:
# Latest Events Agent: Session Setup and Run
APP_NAME = "wanderwise_app"
USER_ID = "user_001"
SESSION_ID = "wanderwise_session_001"

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

runner = Runner(
    agent=latest_events_agent,
    app_name=APP_NAME,
    session_service=session_service
)

In [11]:
# Helper function for querying the agent and displaying results
def ask_events_agent(prompt):
    # Input validation for user safety and agent reliability
    if not validate_city_query(prompt):
        display(Markdown("🚫 **Invalid input detected. Please enter a valid question about events or destinations.**"))
        return

    content = types.Content(role="user", parts=[types.Part(text=prompt)])
    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )
    for event in events:
        if event.is_final_response():
            display(Markdown(f"**Prompt:** {prompt}\n\n---\n\n**Agent Response:**\n\n{event.content.parts[0].text}"))


In [12]:
# Example 1: Food festivals in London next month
ask_events_agent("What food festivals are happening in London next month?")

**Prompt:** What food festivals are happening in London next month?

---

**Agent Response:**

In London next month, you can attend the Taste of London Food Festival in Regent’s Park from June 19-23. Additionally, the Hampton Court Palace Food Festival will be in August.

Source: ldnfashion.com, allevents.in, montybooch.co.uk, forbes.com, designmynight.com


In [13]:
# Example 2: Major art exhibitions in Paris this summer
ask_events_agent("List major art exhibitions in Paris this summer.")

**Prompt:** List major art exhibitions in Paris this summer.

---

**Agent Response:**

Here are some major art exhibitions in Paris this summer:

*   **Louvre Couture: Art and Fashion** at the Musée du Louvre, until July 21, 2025
*   **We Are Here** at the Petit Palais.
*   **Picasso. Art in Motion** at the Atelier des Lumières, until June 29, 2025
*   **David Hockney, 25** at the Fondation Louis Vuitton, from April 9 to August 31, 2025



In [14]:
# Example 3: Tech conferences in San Francisco in 2025
ask_events_agent("Are there any tech conferences in San Francisco in 2025?")

**Prompt:** Are there any tech conferences in San Francisco in 2025?

---

**Agent Response:**

Yes, there are several tech conferences scheduled in San Francisco for 2025:

*   **Game Developers Conference (GDC):** March 17-21, Moscone Center.
*   **TECHSPO San Francisco:** July 21-22, Grand Hyatt Hotel at SFO.
*   **Dreamforce:** October 14-16, multiple locations around Moscone West.
*   **TechCrunch Disrupt:** October 27-29, Moscone West Center.
*   **QCon San Francisco:** November 17-21, Hyatt Regency San Francisco.

In [15]:
# Example 4: Unrelated or invalid input (should be politely handled)
ask_events_agent("What is life?")

**Prompt:** What is life?

---

**Agent Response:**

I am designed to provide information about travel, events, and activities. I am not able to provide a response about the meaning of life.

In [16]:
# Example 5: Unrelated or invalid input (should be politely handled)
ask_events_agent("DROP TABLE users; --")

**Prompt:** DROP TABLE users; --

---

**Agent Response:**

I'm sorry, I'm not able to help with that.


In [17]:
# 🎉 Congratulations!
# You've just built and run an agent that uses the Tavily third-party search tool via LangChain and ADK to fetch live event data for your itinerary.  
# Try modifying the prompt or agent instructions, and see how the results change!